# BioPred Phase 2 -- Candidate Model Benchmarking

## Purpose

This notebook benchmarks candidate model families for BioPred using the locked training artifacts created in Notebook 07 and the incumbent Logistic Regression baseline established in Notebook 08.

Notebook 07 defined the final scaffold-aware training/test split and saved the raw modeling inputs. Notebook 08 established the first modeling baseline using only the locked training set. Notebook 09 continues to use only the training set. The final scaffold-held-out test set is not used here.

The goal is to compare several defensible candidate models against the Logistic Regression incumbent before moving into model narrowing, feature ablation, descriptor pruning, final tuning, or protected test-set evaluation.

This notebook is not intended to deeply tune every model. It is an intentional first-pass benchmark designed to determine whether more complex model families provide enough ranking improvement, weak-fold robustness, or operational value to justify added runtime, artifact size, tuning burden, and deployment complexity.

## Inputs

This notebook uses the split-aware artifacts saved by Notebook 07, including:

* raw training features
* primary training labels
* sensitivity labels
* training metadata
* scaffold assignments
* feature QA flags
* feature schema
* preprocessing policy

This notebook also uses the baseline outputs saved by Notebook 08, including:

* baseline cross-validation results
* baseline cross-validation summary tables

## Outputs

This notebook will save:

* candidate model cross-validation results
* candidate model summary tables
* candidate model decision table
* runtime comparison tables
* reusable benchmark configuration
* notes on which candidate models should move forward

## Objectives

1. Load the locked training artifacts from Notebook 07.

2. Run lightweight alignment guardrails to confirm the training features, labels, metadata, scaffolds, and QA flags share the expected index.

3. Load the Notebook 08 Logistic Regression baseline results as the incumbent reference.

4. Rebuild the same fold-safe preprocessing pipeline used in Notebook 08.

5. Set up scaffold-aware cross-validation using the training scaffolds only.

6. Define explicit model advancement criteria before benchmarking challenger models.

7. Define the challenger model registry:

   * SGDClassifier
   * Random Forest
   * ExtraTrees
   * XGBoost

8. Run the scaffold-aware challenger benchmark.

   For each challenger model and CV fold, fit a fold-safe pipeline, generate validation scores, compute ranking-oriented metrics, and record challenger runtime.

9. Summarize challenger benchmark results at the model level.

   Aggregate fold-level results into model-level summaries including mean performance, fold variance, weak-fold minima, and runtime.

10. Apply staged model advancement gates.

Screen challenger models, select the strongest challenger, and compare it against the Logistic Regression incumbent under the predefined replacement policy.

11. Align incumbent comparison fields for Stage 3.

Derive any missing incumbent weak-fold comparison fields from the existing Notebook 08 fold-level CV results.

12. Save benchmark results, summary tables, model advancement decisions, and reusable benchmark configuration for later notebooks.


## Cross-Validation Scope and Test-Set Protection

Notebook 09 does not use an independent validation set and does not use the final scaffold-held-out test set.

The project has one protected final test set created in Notebook 07. That test set contains scaffold-held-out molecules and remains untouched during Notebook 09.

All model comparison in this notebook occurs inside the locked training set using scaffold-aware cross-validation. During each CV iteration, the training set is temporarily divided into:

* a fold-training subset
* a fold-validation subset

The fold-validation subset is used only to evaluate that fold's model. It is not a permanent validation set, and it is not the protected final test set.

In the code below, variables such as `fold_train_idx`, `fold_valid_idx`, `X_fold_train`, and `X_fold_valid` refer only to temporary cross-validation partitions drawn from the locked training set.

This distinction is important:

* **Training set:** used for scaffold-aware cross-validation and candidate model comparison
* **Fold-validation subset:** temporary held-out portion of the training set inside one CV fold
* **Final test set:** protected scaffold-held-out set reserved for later final evaluation

No model in Notebook 09 is selected, tuned, ranked, or justified using the final test set.

## Model Selection Policy

Notebook 09 uses a staged model selection policy to avoid selecting models based on isolated or post-hoc metric wins.

Average precision is used as the broad ranking qualification metric because BioPred is a ranking-oriented screening project and the active class is the class of operational interest.

Candidate models that pass the broad ranking screen are then compared on early-retrieval utility, including Precision@5%, EF@5%, Precision@10%, and EF@10%.

ROC AUC is retained as a secondary sanity metric for global class separation, but it is not used as the primary model-selection metric.

Runtime is treated as part of model comparison, not as an afterthought. A more complex model should only move forward if its ranking or early-retrieval gains justify additional runtime, dependency burden, artifact size, and deployment complexity.

The Logistic Regression model from Notebook 08 remains the incumbent baseline unless another candidate provides a clear and defensible improvement.



In [1]:
# Force notebook runtime to project root
%cd /home/ryanm/projects/BioPred

from pathlib import Path
import sys
import json
import time
import warnings

# define paths and link src.paths
SRC_DIR = Path.cwd() / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
)

# project imports here, after sys.path setup
import biopred.paths as paths

from biopred.metrics import (
    precision_at_k_percent,
    enrichment_factor_at_k_percent
)

warnings.filterwarnings("ignore")

# edit display options
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)


/home/ryanm/projects/BioPred


In [2]:
# load our locked training artifacts from Notebook 07 (condensed version)

X_train_raw = pd.read_parquet(paths.MODELING_DIR / "X_train_raw.parquet")
y_train = pd.read_parquet(paths.MODELING_DIR / "y_train.parquet")
y_sensitivity = pd.read_parquet(paths.MODELING_DIR / "y_sensitivity_train.parquet")
metadata_train = pd.read_parquet(paths.MODELING_DIR / "metadata_train.parquet")
feature_qa_flags_train = pd.read_parquet(paths.MODELING_DIR / "feature_qa_flags_train.parquet")
split_assignments_train = pd.read_parquet(paths.MODELING_DIR / "split_assignments_train.parquet")

with open(paths.MODELING_DIR / "feature_schema.json", "r") as f:
    feature_schema = json.load(f)

with open(paths.MODELING_DIR / "preprocessing_policy.json", "r") as f:
    preprocessing_policy = json.load(f)

# new here, load our Notebook 08 incumbent baseline results
baseline_cv_results = pd.read_parquet(paths.MODELING_DIR / "baseline_cv_results.parquet")
baseline_cv_summary = pd.read_parquet(paths.MODELING_DIR / "baseline_cv_summary.parquet")



In [3]:
# lightweight standard checks before we begin

# define primary modeling objects
TARGET_COL = "active_median_px_6_0"
SCAFFOLD_COL = "bemis_murcko_scaffold"
RANDOM_STATE = 33  # our locked seed state from previous
N_SPLITS = 5

X = X_train_raw.copy()
y = y_train[TARGET_COL].copy()
groups = split_assignments_train[SCAFFOLD_COL].copy()

rdkit_cols = feature_schema["rdkit_descriptors"]["columns"]
morgan_cols = feature_schema["morgan_fingerprints"]["columns"]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"groups shape: {groups.shape}")
print(f"RDKit descriptor columns: {len(rdkit_cols)}")
print(f"Morgan fingerprint columns: {len(morgan_cols)}")
print(f"Primary active rate: {y.mean():.3f}")

X shape: (1236, 2265)
y shape: (1236,)
groups shape: (1236,)
RDKit descriptor columns: 217
Morgan fingerprint columns: 2048
Primary active rate: 0.776


In [4]:
# quick assertion cell for alignment check

assert X.index.equals(y.index)
assert X.index.equals(split_assignments_train.index)
assert groups.index.equals(X.index)

assert set(rdkit_cols).issubset(X.columns)
assert set(morgan_cols).issubset(X.columns)

guardrail_summary = pd.DataFrame(
    {
        "check" : [

            "n_training_rows",
            "n_features",
            "n_rdkit_cols",
            "n_morgan_cols",
            "n_unique_scaffolds",
            "primary_active_rate",
        ],
        "value" : [
            len(X),
            X.shape[1],
            len(rdkit_cols),
            len(morgan_cols),
            groups.nunique(),
            round(y.mean(), 4),
        ],
    },
)

guardrail_summary

,check,value
0,n_training_rows,1236.0000
1,n_features,2265.0000
2,n_rdkit_cols,217.0000
3,n_morgan_cols,2048.0000
4,n_unique_scaffolds,440.0000
5,primary_active_rate,0.7759


In [5]:
# inspect Notebook 08 baseline artifacts
print(f"baseline_cv_results shape: {baseline_cv_results.shape}")
print(f"baseline_cv_summary shape: {baseline_cv_summary.shape}")

baseline_cv_summary

baseline_cv_results shape: (15, 13)
baseline_cv_summary shape: (3, 16)


valid_active_rate           average_precision             roc_auc           balanced_accuracy           precision_at_5_percent           ef_at_5_percent            \
                                 mean       std              mean       std      mean       std              mean       std                   mean       std            mean       std   
model_name                                                                                                                                                                               
dummy_most_frequent          0.782334  0.066796          0.782334  0.066796  0.500000  0.000000          0.500000  0.000000               0.874402  0.101622        1.121450  0.137239   
logreg_balanced              0.782334  0.066796          0.946253  0.042205  0.858641  0.072279          0.751787  0.078018               0.960000  0.089443        1.234961  0.164492   
logreg_unweighted            0.782334  0.066796          0.946038  0.043070  0.859326  0.072832          0.742450  0.087826               0.960000  0.089443        1.234961  0.164492   

                    precision_at_10_percent           ef_at_10_percent            
                                       mean       std             mean       std  
model_name                                                                        
dummy_most_frequent                0.808960  0.107144         1.040601  0.157569  
logreg_balanced                    0.963158  0.051299         1.236913  0.107016  
logreg_unweighted                  0.963158  0.051299         1.236913  0.107016

In [6]:
# make a cleaner copy, since model_name is our index.
baseline_summary_flat = baseline_cv_summary.copy()

baseline_summary_flat.columns = [
    f"{metric}_{stat}" for metric, stat in baseline_summary_flat.columns
]

baseline_summary_flat = baseline_summary_flat.reset_index()

baseline_summary_flat

,model_name,valid_active_rate_mean,valid_active_rate_std,average_precision_mean,average_precision_std,roc_auc_mean,roc_auc_std,balanced_accuracy_mean,balanced_accuracy_std,precision_at_5_percent_mean,precision_at_5_percent_std,ef_at_5_percent_mean,ef_at_5_percent_std,precision_at_10_percent_mean,precision_at_10_percent_std,ef_at_10_percent_mean,ef_at_10_percent_std
0,dummy_most_frequent,0.782334,0.066796,0.782334,0.066796,0.500000,0.000000,0.500000,0.000000,0.874402,0.101622,1.121450,0.137239,0.808960,0.107144,1.040601,0.157569
1,logreg_balanced,0.782334,0.066796,0.946253,0.042205,0.858641,0.072279,0.751787,0.078018,0.960000,0.089443,1.234961,0.164492,0.963158,0.051299,1.236913,0.107016
2,logreg_unweighted,0.782334,0.066796,0.946038,0.043070,0.859326,0.072832,0.742450,0.087826,0.960000,0.089443,1.234961,0.164492,0.963158,0.051299,1.236913,0.107016


In [7]:
# mark the incumbent model for usage in our upcoming modeling pipeline
INCUMBENT_MODEL_NAME = "logreg_unweighted"

incumbent_baseline = baseline_summary_flat.query(
    "model_name == @INCUMBENT_MODEL_NAME"
).copy()

# quick assertion to make sure name is good and that there is only one match to the name.
assert len(incumbent_baseline) == 1

incumbent_baseline

,model_name,valid_active_rate_mean,valid_active_rate_std,average_precision_mean,average_precision_std,roc_auc_mean,roc_auc_std,balanced_accuracy_mean,balanced_accuracy_std,precision_at_5_percent_mean,precision_at_5_percent_std,ef_at_5_percent_mean,ef_at_5_percent_std,precision_at_10_percent_mean,precision_at_10_percent_std,ef_at_10_percent_mean,ef_at_10_percent_std
2,logreg_unweighted,0.782334,0.066796,0.946038,0.04307,0.859326,0.072832,0.74245,0.087826,0.96,0.089443,1.234961,0.164492,0.963158,0.051299,1.236913,0.107016


In [8]:
# extract the values needed from the summary for our incumbent model to use later
incumbent_row = incumbent_baseline.iloc[0]

INCUMBENT_MEAN_AP = incumbent_row["average_precision_mean"]
INCUMBENT_MEAN_ROC_AUC = incumbent_row["roc_auc_mean"]
INCUMBENT_MEAN_PRECISION_AT_5 = incumbent_row["precision_at_5_percent_mean"]
INCUMBENT_MEAN_EF_AT_5 = incumbent_row["ef_at_5_percent_mean"]
INCUMBENT_MEAN_PRECISION_AT_10 = incumbent_row["precision_at_10_percent_mean"]
INCUMBENT_MEAN_EF_AT_10 = incumbent_row["ef_at_10_percent_mean"]

incumbent_metric_summary = pd.Series(
    {
        "incumbent_model" : INCUMBENT_MODEL_NAME,
        "mean_average_precision" : INCUMBENT_MEAN_AP,
        "mean_roc_auc" : INCUMBENT_MEAN_ROC_AUC,
        "mean_precision_at_5_percent" : INCUMBENT_MEAN_PRECISION_AT_5,
        "mean_ef_at_5_percent" : INCUMBENT_MEAN_EF_AT_5,
        "mean_precision_at_10_percent" : INCUMBENT_MEAN_PRECISION_AT_10,
        "mean_ef_at_10_percent" : INCUMBENT_MEAN_EF_AT_10,
    },
    name = "value",
)

incumbent_metric_summary

incumbent_model                 logreg_unweighted
mean_average_precision                   0.946038
mean_roc_auc                             0.859326
mean_precision_at_5_percent                  0.96
mean_ef_at_5_percent                     1.234961
mean_precision_at_10_percent             0.963158
mean_ef_at_10_percent                    1.236913
Name: value, dtype: object

**Setup Checkpoint — Training Objects and Incumbent Baseline**

The notebook has loaded the locked Notebook 07 training artifacts and the saved Notebook 08 baseline results.

The working modeling objects are now defined:

* `X`: frozen training feature matrix
* `y`: primary training label, `active_median_px_6_0`
* `groups`: Bemis-Murcko scaffold labels for scaffold-aware CV
* `rdkit_cols`: RDKit descriptor columns
* `morgan_cols`: Morgan fingerprint columns

Only lightweight guardrails were applied because detailed artifact validation was already completed in Notebook 08.

The unweighted Logistic Regression model from Notebook 08 has been identified as the incumbent baseline. This model is the reference point for candidate model advancement decisions later in the notebook.

The final scaffold-held-out test set remains unloaded and unused.

### **Section 4: Rebuild Fold-Safe Preprocessing**

This section rebuilds the preprocessing strategy used in Notebook 08.

The preprocessing pipeline must remain fold-safe. RDKit descriptor preprocessing is fit only on the fold-training subset inside each cross-validation iteration. Morgan fingerprint bits are passed through unchanged.

This prevents information from the fold-validation subset from leaking into imputation or scaling.

Preprocessing policy:

* RDKit descriptors: median imputation followed by standard scaling
* Morgan fingerprints: passthrough
* no globally fit preprocessing
* no protected test-set usage

In [9]:
# rebuild the fold-safe preprocessing

rdkit_preprocessor = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy = "median")),
        ("scaler", StandardScaler())
    ]
)

preprocessor = ColumnTransformer(
    transformers = [
        ("rdkit", rdkit_preprocessor, rdkit_cols),
        ("morgan", "passthrough", morgan_cols),
    ],
    remainder = "drop",
    verbose_feature_names_out=False
)

preprocessor

,transformers,"[('rdkit', ...), ('morgan', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


The preprocessing object has been rebuilt without fitting it globally.

This object will be cloned inside each fold-level model pipeline so that imputation and scaling are learned only from the fold-training subset. Morgan fingerprint bits remain unchanged.

### **Section 5: Set Up Scaffold-Aware Cross-Validation**

This section defines the scaffold-aware cross-validation strategy used for candidate benchmarking.

Notebook 09 does not create a separate validation set. Instead, each cross-validation iteration creates a temporary fold-training subset and fold-validation subset from the locked training set.

The fold-validation subset is used only to evaluate that fold's model. It is not the protected final scaffold-held-out test set.

The same scaffold-aware CV policy from Notebook 08 is reused:

* `StratifiedGroupKFold`
* 5 folds
* shuffled folds
* `random_state=33`
* scaffold groups from `bemis_murcko_scaffold`

This keeps candidate model comparison aligned with the incumbent baseline from Notebook 08.

In [10]:
# define scaffold-aware cross-validation strategy
# we will use our structure from Notebook 08, changing names slightly of vars.

SELECTED_SPLIT_SEED = 33

cv = StratifiedGroupKFold(
    n_splits = 5,
    shuffle = True,
    random_state = SELECTED_SPLIT_SEED
)

In [11]:
# audit the scaffold-cv folds, making sure we produced reasonable validation folds

cv_fold_rows = []

# change var names slightly from Notebook 08 structure
for fold_id, (fold_train_idx, fold_valid_idx) in enumerate(cv.split(X, y, groups = groups), start = 1):
    y_train_fold = y.iloc[fold_train_idx]
    y_valid_fold = y.iloc[fold_valid_idx]

    groups_fold_train = groups.iloc[fold_train_idx]
    groups_fold_valid = groups.iloc[fold_valid_idx]

    scaffold_overlap = set(groups_fold_valid).intersection(set(groups_fold_train))

    cv_fold_rows.append(
        {
            "fold" : fold_id,
            "n_train" : len(fold_train_idx),
            "n_valid" : len(fold_valid_idx),
            "valid_n_active" : int(y_valid_fold.sum()),
            "valid_n_inactive" : int((y_valid_fold == 0).sum()),
            "valid_active_rate" : float(y_valid_fold.mean()),
            "train_n_scaffolds" : groups_fold_train.nunique(),
            "valid_n_scaffolds" : groups_fold_valid.nunique(),
            "n_scaffold_overlap" : len(scaffold_overlap)
        }
    )

cv_fold_audit = pd.DataFrame(cv_fold_rows)

assert (cv_fold_audit["n_scaffold_overlap"] == 0).all()
assert cv_fold_audit["valid_n_active"].min() > 0
assert cv_fold_audit["valid_n_inactive"].min() > 0

print("Scaffold-CV fold audit passed.")

cv_fold_audit

Scaffold-CV fold audit passed.


,fold,n_train,n_valid,valid_n_active,valid_n_inactive,valid_active_rate,train_n_scaffolds,valid_n_scaffolds,n_scaffold_overlap
0,1,1047,189,149,40,0.788360,348,92,0
1,2,1001,235,194,41,0.825532,354,86,0
2,3,1004,232,201,31,0.866379,356,84,0
3,4,1028,208,149,59,0.716346,353,87,0
4,5,864,372,266,106,0.715054,349,91,0


The scaffold-aware CV splitter produced five fold-validation subsets from the locked training set.

Each fold contains both active and inactive molecules, and no Bemis-Murcko scaffold appears in both the fold-training and fold-validation subsets. This confirms that candidate models will be compared using scaffold-disjoint validation folds while the final scaffold-held-out test set remains untouched.

### **Section 6: Define Model Advancement Criteria**

This section defines the model advancement policy before candidate benchmarking begins.

Notebook 09 uses a three-stage policy. The first two stages identify the strongest challenger model. The third stage decides whether that challenger is strong enough to replace the Logistic Regression incumbent from Notebook 08.

The incumbent Logistic Regression model is treated as a reference model, not as a challenger in the candidate benchmark.

#### Challenger Model Rationale

Notebook 09 benchmarks four challenger models against the Logistic Regression incumbent from Notebook 08. The incumbent is not included as a challenger because its role is to serve as the reference model in the final incumbent challenge.

The challenger set is intentionally limited to models that test distinct modeling assumptions:


* **SGDClassifier** is included as a sparse-friendly linear challenger. It tests whether an online/large-scale linear optimization approach can match or improve the incumbent while preserving low operational complexity.

* **Random Forest** is included as a bagged nonlinear tree baseline. It tests whether nonlinear feature interactions improve scaffold-CV ranking performance without introducing an external modeling dependency.

* **ExtraTrees** is included as a higher-randomness tree ensemble contrast. It tests whether stronger ensemble randomization improves ranking robustness or weak-fold behavior relative to Random Forest.

* **XGBoost** is included as a regularized boosted-tree challenger. It tests whether a heavier nonlinear model can produce enough ranking or top-K retrieval improvement to justify its additional dependency burden, runtime, and deployment complexity.

This challenger set is not intended to exhaustively tune every possible model family. It is a first-pass benchmark designed to determine whether any non-incumbent model family earns deeper follow-up.

#### Stage 1: Broad Challenger Eligibility

Candidate challengers must first clear an average-precision floor relative to the incumbent baseline.

This stage uses average precision because BioPred is a ranking-oriented screening project. Passing this gate does not mean a challenger is better than the incumbent. It only means the challenger has enough broad scaffold-CV ranking signal to remain under consideration.

Stage 1 does not force a fixed number of models forward. A challenger must first clear the AP eligibility floor. Among eligible challengers, only the top fraction by average precision proceeds to Stage 2. With the planned four-model challenger pool, this keeps at most two challengers for early-retrieval review.

#### Stage 2: Challenger Selection

Eligible challengers are then compared on early-retrieval utility.

Precision@5% is treated as the primary challenger-selection metric because BioPred is designed around top-ranked screening triage. EF@5% is retained as normalized enrichment context.

The output of this stage is the strongest challenger model.

#### Stage 3: Incumbent Challenge

The strongest challenger is then compared directly against the Logistic Regression incumbent.

The incumbent is not replaced by a negligible metric difference. To supplant the incumbent, the challenger must show a practically meaningful top-K advantage while avoiding material degradation in broad ranking quality, weak-fold behavior, runtime, and operational simplicity.

Runtime is treated as a review flag rather than an automatic exclusion rule. A slower or heavier model can still advance, but it must justify its added cost.

These criteria are defined before candidate benchmarking so that model comparison remains disciplined and transparent. The final scaffold-held-out test set remains unused.

The advancement gates do not limit which metrics are computed. All challenger models will still receive the full benchmark metric set used in Notebook 08, including average precision, ROC AUC, balanced accuracy, Precision@K%, EF@K%, and runtime metrics.

The gates only define how those metrics are used for staged model advancement.

In [12]:
# Objective 6A - Stage 1: broad challenger eligibility

# pull the incumbent mean AP from the incumbent summary
incumbent_mean_ap = incumbent_baseline["average_precision_mean"].iloc[0]

# choose the AP eligibility ratio for the first gate, considering 0.85 here against our incumbent value of 0.946
AP_GATE_RATIO = 0.85

# compute the minimum AP a challenger needs to pass Stage 1.
stage_1_ap_floor = incumbent_mean_ap * AP_GATE_RATIO

# choose how many eligible challengers can proceed to Stage 2.
GATE_1_MAX_PASS_MODELS = 2

stage_1_policy = {
    "stage" : "stage_1_broad_challenger_eligibility",
    "metric" : "average_precision_mean",
    "incumbent_model" : INCUMBENT_MODEL_NAME,
    "incumbent_mean_ap" : incumbent_mean_ap,
    "ap_gate_ratio" : AP_GATE_RATIO,
    "stage_1_ap_floor" : stage_1_ap_floor,
    "pool_limiter" : "keep_top_n_eligible_challengers_by_average_precision_mean",
    "gate_1_max_pass_models" : GATE_1_MAX_PASS_MODELS,
    "interpretation" : (
        "Passing Stage 1 indicates sufficient broad ranking signal. "
        "Among eligible challengers, only the top mean-AP performers proceed to Stage 2. "
        "This does not imply superiority over the incumbent. "
    ),
}

pd.Series(stage_1_policy, name = "value")



stage                                                                                                                                                                                stage_1_broad_challenger_eligibility
metric                                                                                                                                                                                             average_precision_mean
incumbent_model                                                                                                                                                                                         logreg_unweighted
incumbent_mean_ap                                                                                                                                                                                                0.946038
ap_gate_ratio                                                                                                                   

In [13]:
# Objective 6B - Stage 2: challenger selection

# choose our primary metric for Gate 2.
TOPK_GATE_METRIC = "precision_at_5_percent_mean"

# select how many models move on from Gate 2.
GATE_2_MAX_PASS_MODELS = 1

# introduce a tie-tolerance threshold, so a model can't win through through a small difference in a single metric
# differences smaller than this are treated as not meaningful
STAGE_2_PRIMARY_TIE_TOLERANCE = 0.015

# for potential tie-breaking scenarios, list metrics to use and in order
STAGE_2_TIE_BREAK_ORDER = [
    "precision_at_10_percent_mean",
    "average_precision_mean",
    "total_fold_time_seconds_mean",
]

# define sort direction for each seconday tie-break metric.
STAGE_2_TIE_BREAK_ASCENDING = [
    False,
    False,
    True,
]

# policy rules
stage_2_policy = {
    "stage" : "stage_2_challenger_selection",
    "primary_metric" : TOPK_GATE_METRIC,
    "primary_tie_tolerance" : STAGE_2_PRIMARY_TIE_TOLERANCE,
    "pool_limiter" : "keep_top_n_stage_1_survivors_by_topk_metric_with_practical_tie_tolerance",
    "gate_2_max_pass_models" : GATE_2_MAX_PASS_MODELS,
    "tie_break_order" : STAGE_2_TIE_BREAK_ORDER,
    "tie_break_ascending" : STAGE_2_TIE_BREAK_ASCENDING,
    "tie_break_interpretation" : (
        "If challengers are within the practical tolerance of Precision@5%, "
        "selection is resolved using mean Precision@10%, mean average precision, "
        " and mean total fold runtime."
    ),
    "reported_context_metrics" : [
        "ef_at_5_percent_mean",
        "ef_at_10_percent_mean",
    ],
    "interpretation" : (
        "Stage 2 selects the strongest early-retrieval challenger from the Stage 1 survivor pool. "
        "This model earns the right to face the incumbent, but it has not yet supplanted it. "
    ),
}

pd.Series(stage_2_policy, name = "value")

stage                                                                                                                                                                        stage_2_challenger_selection
primary_metric                                                                                                                                                                precision_at_5_percent_mean
primary_tie_tolerance                                                                                                                                                                               0.015
pool_limiter                                                                                                                     keep_top_n_stage_1_survivors_by_topk_metric_with_practical_tie_tolerance
gate_2_max_pass_models                                                                                                                                                                          

In [14]:
# Objective 6C - Stage 3: incumbent challenge

# choose the minimum Prec@5% improvement required to dethrone the incumbent model.
# We use 0.03 as a stricter replacement threshold.
# Rationale: 0.015 approximates the minimum practical top-K resolution across 5 folds.
# Requiring 0.03 demands roughly twice that minimum signal before replacing the simpler incumbent.
MIN_PRECISION_AT_5_REPLACEMENT_DELTA = 0.03

# choose the max acceptable AP loss relative to the incumbent.
# We will use 0.01 here
# This prevents a model from marginally winning at top-k while sacrificing broad ranking quality.
MAX_AP_REPLACEMENT_LOSS = 0.01

# choose the max acceptable weak-fold Precision@5% loss relative to the incumbent.
# this prevents replacement if the challenger has materially wose worst-fold top-k behavior.
MAX_PRECISION_AT_5_MIN_LOSS = 0.015

stage_3_policy = {
    "stage" : "stage_3_incumbent_challenge",
    "incumbent_model" : INCUMBENT_MODEL_NAME,
    "primary_replacement_metric" : "precision_at_5_percent_mean",
    "min_precision_at_5_replacement_delta" : MIN_PRECISION_AT_5_REPLACEMENT_DELTA,
    "broad_ranking_guardrail_metric": "average_precision_mean",
    "max_ap_replacement_loss": MAX_AP_REPLACEMENT_LOSS,
    "weak_fold_metric": "precision_at_5_percent_min",
    "max_precision_at_5_min_loss": MAX_PRECISION_AT_5_MIN_LOSS,
    "runtime_metric": "total_fold_time_seconds_mean",
    "runtime_rule": "excluded_from_stage_3_due_to_missing_incumbent_runtime",
    "runtime_note" : (
        "Runtime was used as operational context in the challenger benchmark, "
        "but it is excluded from the incumbent replacement gate because the "
        "Notebook 08 incumbent baseline did not record comparable runtime fields."
    ),
    "interpretation": (
        "The strongest challenger may replace the incumbent only if it shows "
        "a practical mean Precision@5% advantage without unacceptable mean AP degradation, "
        "materially worse weak-fold Precision@5% behavior, or unjustified operational burden."
    ),
}

pd.Series(stage_3_policy, name="value")

stage                                                                                                                                                                                                                                                  stage_3_incumbent_challenge
incumbent_model                                                                                                                                                                                                                                                  logreg_unweighted
primary_replacement_metric                                                                                                                                                                                                                             precision_at_5_percent_mean
min_precision_at_5_replacement_delta                                                                                                                                           

In [15]:
# Objective 6D - combine staged advancement policy

# combine the policies and plans together along with the other expected outcomes for our pending modeling structure.

POLICY_NAME = "notebook_09_staged_model_advancement_policy"

model_advancement_policy = {
    "policy_name" : POLICY_NAME,
    "incumbent_name" : INCUMBENT_MODEL_NAME,
    "incumbent_role" : "reference_not_challenger",
    "challenger_pool_role" : "candidate_models_compared_before_incumbent_challenge",
    "stage_1" : stage_1_policy,
    "stage_2" : stage_2_policy,
    "stage_3" : stage_3_policy,
    "full_metric_collection" : True,
    "metrics_collected_for_all_challengers" : [
        "valid_active_rate",
        "active_precision",
        "roc_auc",
        "balanced_accuracy",
        "precision_at_5_percent",
        "ef_at_5_percent",
        "precision_at_10_percent",
        "ef_at_10_percent",
        "fit_time_seconds",
        "score_time_seconds",
        "total_fold_time_seconds",
    ],
    "roc_auc_role" : "secondary_sanity_metric",
    "runtime_role" : "collected_for_all_models_used_in_stage_3_review",
    "test_set_used" : False,
}

pd.Series(model_advancement_policy, name = "value")

policy_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   notebook_09_staged_model_a

The staged advancement policy is now defined.

Notebook 09 will use this policy to separate challenger discovery from incumbent replacement. All challenger models will receive the full benchmark metric set, but only selected metrics are used at each advancement stage. The final scaffold-held-out test set remains unused.

This advancement policy is specific to the current BioPred target, dataset size, label balance, model pool, and screening objective. If BioPred is expanded to additional targets, larger datasets, broader chemical spaces, lower active-prevalence settings, or a larger model-search program, the policy should be revisited.

In particular, the AP eligibility floor, top-K selection metric, replacement delta, weak-fold criteria, and runtime review threshold may need to change as the data regime and operational objective change.

### **Section 7: Define Challenger Model Registry**

This section defines the challenger model registry used for Notebook 09 benchmarking.

The Logistic Regression model from Notebook 08 is not included in this registry because it is the incumbent reference model. The registry includes only non-incumbent challenger models.

The challenger registry is intentionally lean. Each entry stores the unfitted estimator and a small amount of metadata needed later for benchmark summaries and model-advancement review.

Model rationale was defined in the Section 6 policy discussion and is not repeated here.

### Challenger Registry Design

Each challenger entry includes:

* `estimator`: unfitted model object
* `model_family`: broad model class
* `complexity_tier`: rough operational complexity label

The estimators remain unfitted here. They will be cloned inside each scaffold-CV fold during benchmarking.

In [16]:
# Objective 7A - Define challenger models

challenger_models = {
    "sgd_log_loss" : {
        "estimator" : SGDClassifier(
            loss = "log_loss",
            penalty = "l2",
            max_iter = 5000,
            tol = 1e-3,
            random_state=SELECTED_SPLIT_SEED,
        ),
        "model_family" : "linear",
        "complexity_tier" : "low",
    },

    "random_forest" : {
        "estimator" : RandomForestClassifier(
            n_estimators = 300,
            max_depth = None,
            min_samples_leaf = 2,
            random_state = SELECTED_SPLIT_SEED,
            n_jobs = -1,
            class_weight = None,
        ),
        "model_family" : "tree_ensemble",
        "complexity_tier" : "medium",
    },

    "extra_trees" : {
        "estimator" : ExtraTreesClassifier(
            n_estimators = 300,
            max_depth = None,
            min_samples_leaf = 2,
            random_state = SELECTED_SPLIT_SEED,
            n_jobs = -1,
            class_weight = None,
        ),
        "model_family" : "tree_ensemble",
        "complexity_tier" : "medium",
    },

    "xgboost" : {
        "estimator" : XGBClassifier(
            n_estimators = 300,
            max_depth = 4,
            learning_rate = 0.05,
            subsample = 0.8,
            colsample_by_tree = 0.8,
            eval_metric = "logloss",
            random_state = SELECTED_SPLIT_SEED,
            n_jobs = -1,
        ),
        "model_family" : "boosted_tree",
        "complexity_tier" : "high",
    },
}

list(challenger_models.keys())

['sgd_log_loss', 'random_forest', 'extra_trees', 'xgboost']

The challenger registry is now defined.

The registry contains four unfitted non-incumbent models: SGDClassifier, Random Forest, ExtraTrees, and XGBoost. These estimators will be cloned inside each scaffold-CV fold during benchmarking so that each model is evaluated under the same fold-safe preprocessing and validation framework.

The challenger hyperparameters are intentionally moderate first-pass benchmark settings. They are not presented as optimized final configurations.

The purpose of Notebook 09 is to compare model families under the same scaffold-aware CV framework, not to exhaustively tune each model. Parameter choices therefore emphasize reasonable defaults, controlled stochastic behavior, reproducible benchmarking, and avoiding excessive complexity before a challenger has earned deeper follow-up.

In [17]:
# Objective 7B - summarize the registry
challenger_registry_summary = pd.DataFrame(
    [
        {
            "model_name" : model_name,
            "estimator_class" : spec["estimator"].__class__.__name__,
            "model_family" : spec["model_family"],
            "complexity_tier" : spec["complexity_tier"],
        }
        for model_name, spec in challenger_models.items()
    ]
)

challenger_registry_summary

,model_name,estimator_class,model_family,complexity_tier
0,sgd_log_loss,SGDClassifier,linear,low
1,random_forest,RandomForestClassifier,tree_ensemble,medium
2,extra_trees,ExtraTreesClassifier,tree_ensemble,medium
3,xgboost,XGBClassifier,boosted_tree,high


### **Section 8: Benchmark Challenger Models Under Scaffold-Aware CV**

This section benchmarks the challenger models using the scaffold-aware cross-validation setup defined earlier.

Each model is evaluated on the same folds, with the same fold-safe preprocessing pipeline, and without using the protected final test set. The benchmark loop records fold-level predictions, ranking metrics, and runtime measurements so that later sections can compare both model performance and operational cost.

The goal is not to select a final production model in this section. The goal is to generate a consistent fold-level results table that can be summarized and interpreted using the advancement policy defined in Section 6.

Before running the full benchmark, we will define a small helper for extracting positive-class scores from our fitted classifiers.

Ranking metrics such as average precision, ROC AUC, Precision@K%, and EF@K% require continuous scores rather than hard class labels.

In [18]:
# Objective 8A - define postitive-class scoring helper, for score extraction.

def get_positive_class_scores(fitted_pipeline, X_eval) :
    """
    Return positive-class scores from a fitted classifier pipeline.

    Expected use:
    - fitted_pipeline has already been fit inside one CV fold.
    - X_eval is the fold-validation feature matrix.
    - returned scores are used for ranking metrics
    """

    proba = fitted_pipeline.predict_proba(X_eval)

    positive_class_scores = proba[:, 1]

    return positive_class_scores

In [19]:
# Objective 8B - define fold metric helper, for computing the metrics.

def compute_fold_metrics(y_true, y_score, y_pred) :
    """
    Compute benchmark metrics for one model on one fold-validation subset.
    """

    fold_metrics = {
        "valid_active_rate" : float(y_true.mean()),
        "average_precision" : average_precision_score(y_true, y_score),
        "roc_auc" : roc_auc_score(y_true, y_score),
        "balanced_accuracy" : balanced_accuracy_score(y_true, y_pred),
        "precision_at_5_percent" : precision_at_k_percent(y_true, y_score, k_percent=5),
        "ef_at_5_percent" : enrichment_factor_at_k_percent(y_true, y_score, k_percent=5),
        "precision_at_10_percent" : precision_at_k_percent(y_true, y_score, k_percent=10),
        "ef_at_10_percent" : enrichment_factor_at_k_percent(y_true, y_score, k_percent=10),
    }

    return fold_metrics

In [20]:
# Objective 8C - benchmark one model on one fold (as a test before full workflow with all challengers)

test_model_name = "sgd_log_loss"

# retrieve the model specs for our test model
test_model_spec = challenger_models[test_model_name]

# get the first scaffold-CV fold.
fold_id = 1
fold_train_idx, fold_valid_idx = next(cv.split(X, y, groups = groups))

# create fold-level train/validation objects.
X_fold_train = X.iloc[fold_train_idx]
X_fold_valid = X.iloc[fold_valid_idx]

y_fold_train = y.iloc[fold_train_idx]
y_fold_valid = y.iloc[fold_valid_idx]

# build a fold-safe model_pipeline, using existing pipeline build. and estimator data from model build.
fold_pipeline = Pipeline(
    steps = [
        ("preprocessor", clone(preprocessor)),
        ("model", clone(test_model_spec["estimator"])),
    ]
)

# fit the pipeline and record fit time.
fit_start_time = time.perf_counter()
fold_pipeline.fit(X_fold_train, y_fold_train)
fit_time_seconds = time.perf_counter() - fit_start_time

# extract positive-class scores and record start time
score_start_time = time.perf_counter()
y_score = get_positive_class_scores(fold_pipeline, X_fold_valid)
score_time_seconds = time.perf_counter() - score_start_time

# get the hard class predictions for balanced accuracy.
y_pred = fold_pipeline.predict(X_fold_valid)

# compute fold metrics
fold_metrics = compute_fold_metrics(
    y_true = y_fold_valid,
    y_score = y_score,
    y_pred = y_pred,
)

# combine model metadata, fold metadata, runtime, and metrics
single_fold_result = {
    "model_name" : test_model_name,
    "fold" : fold_id,
    "model_family" : test_model_spec["model_family"],
    "complexity_tier" : test_model_spec["complexity_tier"],
    "n_fold_train" : len(fold_train_idx),
    "n_fold_valid" : len(fold_valid_idx),
    "fit_time_seconds" : fit_time_seconds,
    "score_time_seconds" : score_time_seconds,
    "total_fold_time_seconds" : fit_time_seconds + score_time_seconds, **fold_metrics,
}

pd.Series(single_fold_result, name = "value")





model_name                 sgd_log_loss
fold                                  1
model_family                     linear
complexity_tier                     low
n_fold_train                       1047
n_fold_valid                        189
fit_time_seconds               0.149047
score_time_seconds             0.005473
total_fold_time_seconds         0.15452
valid_active_rate               0.78836
average_precision              0.921996
roc_auc                        0.814681
balanced_accuracy              0.713087
precision_at_5_percent              1.0
ef_at_5_percent                1.268456
precision_at_10_percent             1.0
ef_at_10_percent               1.268456
Name: value, dtype: object

The single-fold benchmark test completed successfully.

This confirms that the challenger registry, fold-safe preprocessing pipeline, score extraction helper, metric helper, and runtime tracking work together for one model on one scaffold-CV fold.

The single-fold result is used only as a workflow check. Model comparison will be based on all challenger models across all scaffold-CV folds.

#### Objective 8D — Run Full Challenger CV Benchmark

This step expands the single-fold benchmark workflow to all challenger models and all scaffold-CV folds.

Each challenger is cloned before fitting, and preprocessing is cloned inside each fold-level pipeline. This keeps the evaluation fold-safe and prevents fitted preprocessing state from leaking across folds or models.

The output is a fold-level benchmark table with one row per model-fold combination.

In [21]:
# building the modeling workflow for all challenger models across all folds now.
# code will look almost the same except building it inside a nested for loop and output will be a df.

benchmark_rows = []

for model_name, model_spec in challenger_models.items():
    print(f"Benchmarking: {model_name}")

    for fold_id, (fold_train_idx, fold_valid_idx) in enumerate(
        cv.split(X, y, groups = groups),
        start = 1
    ):
        # create fold-level train/validation objects.
        X_fold_train = X.iloc[fold_train_idx]
        X_fold_valid = X.iloc[fold_valid_idx]

        y_fold_train = y.iloc[fold_train_idx]
        y_fold_valid = y.iloc[fold_valid_idx]

        # build a fresh pipeline, but still clone the preprocessor and the model estimator.
        fold_pipeline = Pipeline(
            steps = [
                ("preprocessor", clone(preprocessor)),
                ("model", clone(model_spec["estimator"])),
            ]
        )

        # fit the model and record fit time(s)
        fit_start_time = time.perf_counter()
        fold_pipeline.fit(X_fold_train, y_fold_train)
        fit_time_seconds = time.perf_counter() - fit_start_time

        # score the validation fold
        score_start_time = time.perf_counter()
        y_score = get_positive_class_scores(fold_pipeline, X_fold_valid)
        score_time_seconds = time.perf_counter() - score_start_time

        # generate hard predictions for balanced accuracy.
        y_pred = fold_pipeline.predict(X_fold_valid)

        # compute the fold metrics
        fold_metrics = compute_fold_metrics(
            y_true = y_fold_valid,
            y_score = y_score,
            y_pred = y_pred,
        )

        # bring everything together and append to benchmark_rows
        benchmark_rows.append(
            {
                "model_name" : model_name,
                "fold" : fold_id,
                "model_family" : model_spec["model_family"],
                "complexity_tier" : model_spec["complexity_tier"],
                "n_fold_train" : len(fold_train_idx),
                "n_fold_valid" : len(fold_valid_idx),
                "fit_time_seconds" : fit_time_seconds,
                "score_time_seconds" : score_time_seconds,
                "total_fold_time_seconds" : fit_time_seconds + score_time_seconds, **fold_metrics,
            }
        )

candidate_model_cv_results = pd.DataFrame(benchmark_rows)

candidate_model_cv_results

    

Benchmarking: sgd_log_loss
Benchmarking: random_forest
Benchmarking: extra_trees
Benchmarking: xgboost


,model_name,fold,model_family,complexity_tier,n_fold_train,n_fold_valid,fit_time_seconds,score_time_seconds,total_fold_time_seconds,valid_active_rate,average_precision,roc_auc,balanced_accuracy,precision_at_5_percent,ef_at_5_percent,precision_at_10_percent,ef_at_10_percent
0,sgd_log_loss,1,linear,low,1047,189,0.139251,0.005594,0.144845,0.788360,0.921996,0.814681,0.713087,1.000000,1.268456,1.000000,1.268456
1,sgd_log_loss,2,linear,low,1001,235,0.126791,0.019348,0.146138,0.825532,0.927497,0.785957,0.730827,0.833333,1.009450,0.875000,1.059923
2,sgd_log_loss,3,linear,low,1004,232,0.133144,0.005921,0.139065,0.866379,0.904986,0.647087,0.622693,0.583333,0.673300,0.708333,0.817579
3,sgd_log_loss,4,linear,low,1028,208,0.146001,0.006010,0.152011,0.716346,0.960432,0.932772,0.893527,0.909091,1.269067,0.952381,1.329498
4,sgd_log_loss,5,linear,low,864,372,0.182847,0.012440,0.195287,0.715054,0.851523,0.755001,0.702263,0.947368,1.324891,0.921053,1.288089
5,random_forest,1,tree_ensemble,medium,1047,189,0.465378,0.047653,0.513031,0.788360,0.965151,0.882215,0.689933,1.000000,1.268456,1.000000,1.268456
6,random_forest,2,tree_ensemble,medium,1001,235,0.416360,0.053267,0.469627,0.825532,0.969742,0.887101,0.667463,1.000000,1.211340,1.000000,1.211340
7,random_forest,3,tree_ensemble,medium,1004,232,0.407325,0.055539,0.462865,0.866379,0.976511,0.878671,0.672444,1.000000,1.154229,1.000000,1.154229
8,random_forest,4,tree_ensemble,medium,1028,208,0.412026,0.053171,0.465196,0.716346,0.985693,0.962348,0.822205,1.000000,1.395973,1.000000,1.395973
9,random_forest,5,tree_ensemble,medium,864,372,0.404745,0.074114,0.478859,0.715054,0.899867,0.766704,0.502944,1.000000,1.398496,0.947368,1.324891


The challenger benchmark produced 20 fold-level results: four challenger models evaluated across five scaffold-aware CV folds.

The timing and metric patterns now differ across model families, which confirms that each challenger estimator is being fit and evaluated separately. `sgd_log_loss` is the fastest model, while `xgboost` has the highest fit-time burden. `random_forest` and `extra_trees` sit between those two extremes.

Across folds, the tree-based challengers generally show stronger top-K retrieval behavior than the SGD linear challenger, with many folds reaching perfect Precision@5%. XGBoost also shows strong top-K performance, but with higher runtime cost.

The fold-level table should not be used to select a model directly. Its purpose is to provide the raw benchmark evidence. The next step is to summarize these fold-level results into model-level means, standard deviations, and weak-fold minima before applying the staged advancement policy.

### **Section 9: Summarize Challenger Benchmark Metrics**

This section summarizes the fold-level challenger benchmark results at the model level.

The benchmark table from Section 8 contains one row per model-fold combination. This section aggregates those rows to compare each challenger on central performance, fold-to-fold variability, weak-fold behavior, and runtime.

The resulting model-level summary table is the input to the staged advancement policy defined in Section 6.

In [22]:
# summarize challenger benchmark results
# agg by mean, std, and min
metric_cols = [
    "valid_active_rate",
    "average_precision",
    "roc_auc",
    "balanced_accuracy",
    "precision_at_5_percent",
    "ef_at_5_percent",
    "precision_at_10_percent",
    "ef_at_10_percent",
    "fit_time_seconds",
    "score_time_seconds",
    "total_fold_time_seconds",
]

candidate_model_cv_summary = (
    candidate_model_cv_results
    .groupby(["model_name"])[metric_cols]
    .agg(["mean", "std", "min"])
)

# flatten for better view
candidate_model_summary_flat = candidate_model_cv_summary.copy()

candidate_model_summary_flat.columns = [
    f"{metric}_{stat}" for metric, stat in candidate_model_summary_flat.columns
]

candidate_model_summary_flat = candidate_model_summary_flat.reset_index()

candidate_model_summary_flat

,model_name,valid_active_rate_mean,valid_active_rate_std,valid_active_rate_min,average_precision_mean,average_precision_std,average_precision_min,roc_auc_mean,roc_auc_std,roc_auc_min,balanced_accuracy_mean,balanced_accuracy_std,balanced_accuracy_min,precision_at_5_percent_mean,precision_at_5_percent_std,precision_at_5_percent_min,ef_at_5_percent_mean,ef_at_5_percent_std,ef_at_5_percent_min,precision_at_10_percent_mean,precision_at_10_percent_std,precision_at_10_percent_min,ef_at_10_percent_mean,ef_at_10_percent_std,ef_at_10_percent_min,fit_time_seconds_mean,fit_time_seconds_std,fit_time_seconds_min,score_time_seconds_mean,score_time_seconds_std,score_time_seconds_min,total_fold_time_seconds_mean,total_fold_time_seconds_std,total_fold_time_seconds_min
0,extra_trees,0.782334,0.066796,0.715054,0.960229,0.038644,0.892496,0.878858,0.085180,0.738332,0.688579,0.114193,0.502944,1.000000,0.000000,1.000000,1.285699,0.109538,1.154229,0.994737,0.011769,0.973684,1.278338,0.100965,1.154229,0.309267,0.004064,0.302324,0.059392,0.008804,0.053485,0.368659,0.005371,0.362489
1,random_forest,0.782334,0.066796,0.715054,0.959393,0.034163,0.899867,0.875408,0.069944,0.766704,0.670998,0.113445,0.502944,1.000000,0.000000,1.000000,1.285699,0.109538,1.154229,0.989474,0.023538,0.947368,1.270978,0.094504,1.154229,0.421167,0.025113,0.404745,0.056749,0.010133,0.047653,0.477916,0.020561,0.462865
2,sgd_log_loss,0.782334,0.066796,0.715054,0.913287,0.039950,0.851523,0.787100,0.103219,0.647087,0.732480,0.099098,0.622693,0.854625,0.163366,0.583333,1.109033,0.272672,0.673300,0.891353,0.111998,0.708333,1.152709,0.214418,0.817579,0.145607,0.022004,0.126791,0.009862,0.006025,0.005594,0.155469,0.022729,0.139065
3,xgboost,0.782334,0.066796,0.715054,0.955421,0.037777,0.889067,0.871062,0.070380,0.755533,0.725091,0.119646,0.559583,0.989474,0.023538,0.947368,1.270978,0.094504,1.154229,0.981140,0.026114,0.947368,1.260883,0.104621,1.154229,1.192830,0.157489,0.943590,0.007677,0.001036,0.006438,1.200507,0.158456,0.950028


### **Section 10: Apply Stage 1 and Stage 2 Advancement Gates**

This section applies the predefined advancement policy to the challenger model summary table.

Stage 1 screens challenger models using mean average precision. A model must clear the AP eligibility floor relative to the Logistic Regression incumbent before it can continue. Among eligible models, only the top challengers by mean AP proceed.

Stage 2 selects the strongest early-retrieval challenger from the Stage 1 survivors using mean Precision@5%. Tie-breaking uses additional predefined summary metrics rather than manual selection.

The output of this section is the primary challenger model that will be compared against the incumbent in the final incumbent-challenge step.

In [23]:
# Objective 10A - apply Stage 1 AP eligibility gate

# copy our candidate_model_sumamry_flat for usage here
stage_1_results = candidate_model_summary_flat.copy()

# compare ap_mean against stage_1_ap_floor
stage_1_results["stage_1_ap_eligible"] = (
    stage_1_results["average_precision_mean"] >= stage_1_ap_floor
)

# rank eligible models by mean AP
stage_1_eligible = (
    stage_1_results
    .query("stage_1_ap_eligible")
    .sort_values("average_precision_mean", ascending = False)
    .copy()
)

# keep only the top eligible challengers allowed by the Stage 1 policy
stage_1_survivors = stage_1_eligible.head(GATE_1_MAX_PASS_MODELS).copy()

# add a flag back to the full Stage 1 table
stage_1_results["stage_1_selected"] = stage_1_results["model_name"].isin(
    stage_1_survivors["model_name"]
)

stage_1_results[
    [
        "model_name",
        "average_precision_mean",
        "average_precision_min",
        "stage_1_ap_eligible",
        "stage_1_selected",
    ]
].sort_values("average_precision_mean", ascending = False)

,model_name,average_precision_mean,average_precision_min,stage_1_ap_eligible,stage_1_selected
0,extra_trees,0.960229,0.892496,True,True
1,random_forest,0.959393,0.899867,True,True
3,xgboost,0.955421,0.889067,True,False
2,sgd_log_loss,0.913287,0.851523,True,False


Stage 1 selected `extra_trees` and `random_forest` as the broad-ranking challenger survivors.

All four challenger models cleared the AP eligibility floor, but only the top two eligible models by mean average precision advanced. `xgboost` performed strongly but did not exceed the tree-ensemble challengers on mean AP, and therefore did not advance under the predefined Stage 1 policy.

This is a useful result: the heavier boosted-tree model did not automatically outperform simpler scikit-learn tree ensembles under scaffold-aware CV. Further XGBoost tuning could be explored later, but this first-pass benchmark does not prioritize it for the incumbent challenge.

In [24]:
# Objective 10B - apply Stage 2 challenger-selection gate (see policy in Section 6)

# start from Stage 1 survivors
stage_2_input = stage_1_survivors.copy()

# find the best primary Stage 2 score
best_stage_2_primary_score = stage_2_input[TOPK_GATE_METRIC].max()

# keep models within the practical tie tolerance of the best primary score
stage_2_practical_tie_pool = (
    stage_2_input
    .loc[
        stage_2_input[TOPK_GATE_METRIC] >= (
            best_stage_2_primary_score - STAGE_2_PRIMARY_TIE_TOLERANCE
        )
    ]
)

# sort the practical-tie pool using secondary tie-break rules.
stage_2_ranked = (
    stage_2_practical_tie_pool
    .sort_values(
        by = STAGE_2_TIE_BREAK_ORDER,
        ascending = STAGE_2_TIE_BREAK_ASCENDING,
    )
    .copy()
)

# select the top model after practical tie handling
stage_2_selected = stage_2_ranked.head(GATE_2_MAX_PASS_MODELS).copy()

# add a selection flag
stage_2_ranked["stage_2_selected"] = stage_2_ranked["model_name"].isin(
    stage_2_selected["model_name"]
)

# build the display
stage_2_cols = [
    
        "model_name",
        "precision_at_5_percent_mean",
        "precision_at_10_percent_mean",
        "average_precision_mean",
        "total_fold_time_seconds_mean",
        "stage_2_selected",
]

stage_2_ranked[stage_2_cols]



,model_name,precision_at_5_percent_mean,precision_at_10_percent_mean,average_precision_mean,total_fold_time_seconds_mean,stage_2_selected
0,extra_trees,1.0,0.994737,0.960229,0.368659,True
1,random_forest,1.0,0.989474,0.959393,0.477916,False


Stage 2 selected ExtraTrees as the primary challenger.

ExtraTrees and Random Forest were practically tied on mean Precision@5%, with both achieving 1.00 across the Stage 2 comparison. The predefined secondary tie-breaks favored ExtraTrees: it had higher mean Precision@10%, slightly higher mean average precision, and lower mean fold runtime.

This selection does not replace the Logistic Regression incumbent. It identifies ExtraTrees as the strongest challenger to carry into the incumbent-challenge comparison.

In [25]:
# Objective 10C - identify our primary challenger
assert len(stage_2_selected == 1)

PRIMARY_CHALLENGER_MODEL_NAME = stage_2_selected["model_name"].iloc[0]

primary_challenger_summary = stage_2_selected.copy()

PRIMARY_CHALLENGER_MODEL_NAME

'extra_trees'

In [26]:
# Objective 10D - compare primary challenger against incumbent

# extract the incumbent model-level summary row
incumbent_comparison = incumbent_baseline.copy()

# label the incumbent row
incumbent_comparison["comparison_role"] = "incumbent"

# extract the primary challenger model-level summary row.
challenger_comparison = primary_challenger_summary.copy()

# label the challenger row
challenger_comparison["comparison_role"] = "challenger"

# combine incumbent and challenger into one comparison table
stage_3_comparison = pd.concat(
    [
        incumbent_comparison,
        challenger_comparison,
    ],
    axis = 0,
    ignore_index=True,
)

# choose the main comparison columns
stage_3_comparison_cols = [
    "comparison_role",
    "model_name",
    "precision_at_5_percent_mean",
    "precision_at_5_percent_min",
    "average_precision_mean",
    "average_precision_min",
    "total_fold_time_seconds_mean",
]

stage_3_comparison[stage_3_comparison_cols]

,comparison_role,model_name,precision_at_5_percent_mean,precision_at_5_percent_min,average_precision_mean,average_precision_min,total_fold_time_seconds_mean
0,incumbent,logreg_unweighted,0.96,NaN,0.946038,NaN,NaN
1,challenger,extra_trees,1.00,1.0,0.960229,0.892496,0.368659


The incumbent summary loaded from Notebook 08 does not contain every model-level field needed for the Stage 3 replacement policy. Rather than rerunning the incumbent model, this section rebuilds the missing incumbent summary fields from the existing Notebook 08 fold-level CV results. This preserves the original scaffold-CV evidence while aligning the incumbent row with the challenger summary format used in Notebook 09.

In [27]:
# check our original baseline_cv_results
baseline_cv_results.columns

Index(['model_name', 'fold', 'n_train', 'n_valid', 'valid_active_rate', 'average_precision', 'roc_auc', 'balanced_accuracy', 'precision_at_5_percent', 'ef_at_5_percent', 'precision_at_10_percent',
       'ef_at_10_percent', 'average_precision_lift_over_prevalence'],
      dtype='object')

In [28]:
# quick fix here, we need to compute the missing metrics for the incumbent
# note: edited the Gate 3 Policy to remove time so it won't be measured for this phase
incumbent_stage_3_summary = incumbent_baseline.copy()


incumbent_fold_results = (
    baseline_cv_results
    .query("model_name  == @INCUMBENT_MODEL_NAME")
    .copy()
)

# compute only the missing weak-fold minimum metrics needed for the Stage 3 gate
incumbent_stage_3_summary["precision_at_5_percent_min"] = (
    incumbent_fold_results["precision_at_5_percent"].min()
)

incumbent_stage_3_summary["average_precision_min"] = (
    incumbent_fold_results["average_precision"].min()
)

# gather the display
incumbent_stage_3_cols = [
    
        "model_name",
        "precision_at_5_percent_mean",
        "precision_at_5_percent_min",
        "average_precision_mean",
        "average_precision_min",
    ]


incumbent_stage_3_summary[incumbent_stage_3_cols]

,model_name,precision_at_5_percent_mean,precision_at_5_percent_min,average_precision_mean,average_precision_min
2,logreg_unweighted,0.96,0.8,0.946038,0.883605


In [29]:
#Objective 10E - compare primary challenger against our incumbent

# Objective 10E — compare primary challenger against incumbent
# we will just redo the build code from above

# use the patched incumbent summary from Objective 10D
incumbent_comparison = incumbent_stage_3_summary.copy()
incumbent_comparison["comparison_role"] = "incumbent"

# use the Stage 2 selected challenger summary
challenger_comparison = primary_challenger_summary.copy()
challenger_comparison["comparison_role"] = "primary_challenger"

# combine into one comparison table
stage_3_comparison = pd.concat(
    [
        incumbent_comparison,
        challenger_comparison,
    ],
    axis=0,
    ignore_index=True,
)

# display policy-aligned comparison columns
stage_3_comparison_cols = [
    "comparison_role",
    "model_name",
    "precision_at_5_percent_mean",
    "precision_at_5_percent_min",
    "average_precision_mean",
    "average_precision_min",
]

stage_3_comparison[stage_3_comparison_cols]

,comparison_role,model_name,precision_at_5_percent_mean,precision_at_5_percent_min,average_precision_mean,average_precision_min
0,incumbent,logreg_unweighted,0.96,0.8,0.946038,0.883605
1,primary_challenger,extra_trees,1.00,1.0,0.960229,0.892496


In [30]:
# Objective 10F - while the visual results look apparent, we still need to apply the Stage 3 gates from our policy.
# this is so we can enact our policy as well as be more definitive with our choice.

# extract scalar rows
incumbent_row = (
    stage_3_comparison
    .query("comparison_role == 'incumbent'")
    .iloc[0]
)

challenger_row = (
    stage_3_comparison
    .query("comparison_role == 'primary_challenger'")
    .iloc[0]
)

# compute the replacement deltas
precision_at_5_mean_delta = (
    challenger_row["precision_at_5_percent_mean"]
    - incumbent_row["precision_at_5_percent_mean"]
)

ap_mean_delta = (
    challenger_row["average_precision_mean"]
    - incumbent_row["average_precision_mean"]
)

precision_at_5_min_delta = (
    challenger_row["precision_at_5_percent_min"]
    - incumbent_row["precision_at_5_percent_min"]
)

# apply Stage 3 gates
passes_precision_at_5_replacement_gate = (
    precision_at_5_mean_delta >= MIN_PRECISION_AT_5_REPLACEMENT_DELTA
)

passes_ap_guardrail = (
    ap_mean_delta >= -MAX_AP_REPLACEMENT_LOSS
)

passes_weak_fold_guardrail = (
    precision_at_5_mean_delta >= -MAX_PRECISION_AT_5_MIN_LOSS
)

challenger_replaces_incumbent = (
    passes_precision_at_5_replacement_gate
    and passes_ap_guardrail
    and passes_weak_fold_guardrail
)

stage_3_decision = pd.DataFrame(
    [
        {
            "incumbent_model" : incumbent_row["model_name"],
            "challenger_model" : challenger_row["model_name"],
            "precision_at_5_mean_delta" : precision_at_5_mean_delta,
            "required_precision_at_5_delta" : MIN_PRECISION_AT_5_REPLACEMENT_DELTA,
            "passes_precision_at_5_replacement_gate" : passes_precision_at_5_replacement_gate,
            "ap_mean_delta" : ap_mean_delta,
            "max_allowed_ap_loss" : MAX_AP_REPLACEMENT_LOSS,
            "passes_ap_guardrail" : passes_ap_guardrail,
            "precision_at_5_min_delta" : precision_at_5_min_delta,
            "max_allowed_precision_at_5_min_loss" : MAX_PRECISION_AT_5_MIN_LOSS,
            "passes_weak_fold_guardrail" : passes_weak_fold_guardrail,
            "challenger_replaces_incumbent" : challenger_replaces_incumbent,
        }
    ]
)

stage_3_decision

,incumbent_model,challenger_model,precision_at_5_mean_delta,required_precision_at_5_delta,passes_precision_at_5_replacement_gate,ap_mean_delta,max_allowed_ap_loss,passes_ap_guardrail,precision_at_5_min_delta,max_allowed_precision_at_5_min_loss,passes_weak_fold_guardrail,challenger_replaces_incumbent
0,logreg_unweighted,extra_trees,0.04,0.03,True,0.014191,0.01,True,0.2,0.015,True,True


#### **Stage 3 Result: Selected CV-Stage Model**

ExtraTrees replaces Logistic Regression as the selected cross-validation-stage model under the predefined Stage 3 replacement policy.

This decision was not based on a simple leaderboard ranking. ExtraTrees first advanced through the challenger benchmark, then had to clear a stricter incumbent-replacement standard against the Logistic Regression baseline. It passed all Stage 3 gates:

* mean Precision@5% improvement exceeded the required replacement threshold;
* mean average precision improved rather than degraded;
* weak-fold Precision@5% behavior improved materially rather than weakening.

As a result, ExtraTrees becomes the subject model of choice for the next modeling steps. It will be used as the primary model family for any targeted follow-up tuning, model artifact creation, ranked output generation, and downstream decision-support analysis in the remainder of the project.

This decision remains internal to scaffold-aware cross-validation on the training set. The final scaffold-held-out test set has not been used. ExtraTrees is therefore the selected CV-stage model, not yet a final test-confirmed model.


**Section 11 was completed within the Stage 3 comparison workflow by aligning the incumbent comparison fields needed for the replacement decision.**

### **Section 12: Save Benchmark and Model-Selection Artifacts**

This section saves the Notebook 09 benchmark outputs and model-selection decisions for downstream use.

The saved artifacts include fold-level challenger results, model-level benchmark summaries, staged gate outputs, the incumbent-vs-challenger comparison, the final CV-stage model-selection decision, and reusable benchmark configuration. These artifacts preserve the evidence trail behind the ExtraTrees selection and allow later notebooks to consume the selected model decision without re-running the full challenger benchmark.

The final scaffold-held-out test set remains unused at this point. Saved outputs from this notebook represent scaffold-aware training-CV evidence only.


In [31]:
# choose our output dir for our artifacts, creating a new folder for these items
benchmark_artifact_dir = paths.MODELING_DIR / "model_benchmarking"
benchmark_artifact_dir.mkdir(parents = True, exist_ok=True)

benchmark_artifact_dir


PosixPath('/home/ryanm/projects/BioPred/data/processed/modeling/model_benchmarking')

In [32]:
# save tabular benchmark and decision artifacts
# parquet files first with confirmation
candidate_model_cv_results.to_parquet(
    benchmark_artifact_dir / "candidate_model_cv_results.parquet",
    index=False,
)

candidate_model_summary_flat.to_parquet(
    benchmark_artifact_dir / "candidate_model_summary_flat.parquet",
    index=False,
)

stage_1_results.to_parquet(
    benchmark_artifact_dir / "stage_1_results.parquet",
    index=False,
)

stage_2_ranked.to_parquet(
    benchmark_artifact_dir / "stage_2_ranked.parquet",
    index=False,
)

stage_2_selected.to_parquet(
    benchmark_artifact_dir / "stage_2_selected.parquet",
    index=False,
)

stage_3_comparison.to_parquet(
    benchmark_artifact_dir / "stage_3_comparison.parquet",
    index=False,
)

stage_3_decision.to_parquet(
    benchmark_artifact_dir / "stage_3_decision.parquet",
    index=False,
)

#confirm files
sorted(path.name for path in benchmark_artifact_dir.iterdir())


['candidate_model_cv_results.parquet',
 'candidate_model_summary_flat.parquet',
 'model_benchmark_config.json',
 'stage_1_results.parquet',
 'stage_2_ranked.parquet',
 'stage_2_selected.parquet',
 'stage_3_comparison.parquet',
 'stage_3_decision.parquet']

In [33]:
# save one json config now, for reusable benchmark configuration

# will make a helper function here for json.dump, so all the different types of values
# get loaded properly into json
def json_load_valid(value):
    if isinstance(value, dict):
        return {key: json_load_valid(val) for key, val in value.items()}
    if isinstance(value, list):
        return [json_load_valid(item) for item in value]
    if isinstance(value, tuple):
        return [json_load_valid(item) for item in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    
    return value

In [34]:
# define selected CV-stage model from the Stage 3 decision
# this is needed for reproducability
selected_cv_stage_model_name = (
    stage_3_decision["challenger_model"].iloc[0]
    if stage_3_decision["challenger_replaces_incumbent"].iloc[0]
    else stage_3_decision["incumbent_model"].iloc[0]
)

selected_cv_stage_model_name

'extra_trees'

In [35]:
# now save the reusable benchmark config and the selected-model decision context
# using our json helper function

model_benchmark_config = {
    "notebook" : "09_candidate_model_benchmarking",
    "selection_scope" : "scaffold_aware_training_policy",
    "final_test_status" : "not_used",
    "selected_split_seed" : SELECTED_SPLIT_SEED,
    "n_splits" : N_SPLITS,
    "target_column" : TARGET_COL,
    "scaffold_column" : SCAFFOLD_COL,
    "incumbent_model_name" : INCUMBENT_MODEL_NAME,
    "primary_challenger_model_name" : PRIMARY_CHALLENGER_MODEL_NAME,
    "selected_cv_stage_model_name" : selected_cv_stage_model_name,
    "challenger_replaces_incumbent" : bool(
        stage_3_decision["challenger_replaces_incumbent"].iloc[0]
    ),
    "challenger_model_names" : list(challenger_models.keys()),
    "feature_policy" : {
        "n_rdkit_descriptor_columns" : len(rdkit_cols),
        "n_morgan_fingerprint_columns" : len(morgan_cols),
        "rdkit_descriptor_columns" : list(rdkit_cols),
        "morgan_fingerprint_columns" : list(morgan_cols),
    },
    "stage_1_policy" : stage_1_policy,
    "stage_2_policy" : stage_2_policy,
    "stage_3_policy" : stage_3_policy,
}

model_benchmark_config = json_load_valid(model_benchmark_config)

with open(
    benchmark_artifact_dir / "model_benchmark_config.json",
    "w",
    encoding = "utf-8",
) as f:
    json.dump(model_benchmark_config, f, indent = 2)


In [36]:
# one more quick confirm for those artifacts
# even though we just did it separately, important information we just stored!

saved_benchmark_artifacts = sorted(
    path.name for path in benchmark_artifact_dir.iterdir()
)

saved_benchmark_artifacts  # Expecting all 8 files we just made

['candidate_model_cv_results.parquet',
 'candidate_model_summary_flat.parquet',
 'model_benchmark_config.json',
 'stage_1_results.parquet',
 'stage_2_ranked.parquet',
 'stage_2_selected.parquet',
 'stage_3_comparison.parquet',
 'stage_3_decision.parquet']

Notebook 09 benchmark artifacts were saved successfully.

The saved outputs preserve the full model-benchmarking evidence trail: fold-level challenger results, model-level summaries, staged advancement decisions, the incumbent-vs-challenger comparison, and the final CV-stage model-selection decision. ExtraTrees is recorded as the selected CV-stage model under scaffold-aware training cross-validation.

The final scaffold-held-out test set remains unused. Downstream notebooks can now consume the selected model decision and benchmark artifacts without re-running the full benchmark workflow.

### **Notebook 09 Summary and Handoff**

Notebook 09 completed the scaffold-aware candidate model benchmarking workflow using the locked training artifacts from Notebook 07 and the Logistic Regression baseline from Notebook 08 as the incumbent reference.

The benchmark evaluated four challenger model families under scaffold-aware cross-validation:

* SGDClassifier
* Random Forest
* ExtraTrees
* XGBoost

Each challenger was evaluated with fold-safe preprocessing and ranking-oriented metrics, including average precision, ROC-AUC, Precision@K, and EF@K. Challenger runtime was also recorded as operational context for the benchmark phase.

Model advancement was governed through a staged policy:

* Stage 1 screened challengers for sufficient broad ranking signal.
* Stage 2 selected the strongest early-retrieval challenger.
* Stage 3 compared the strongest challenger against the Logistic Regression incumbent under a stricter replacement policy.

ExtraTrees was selected as the CV-stage model. It replaced the Logistic Regression incumbent after exceeding the required mean Precision@5% replacement threshold, improving mean average precision, and showing stronger weak-fold Precision@5% behavior.

Saved artifacts include fold-level benchmark results, model-level summaries, staged advancement outputs, incumbent-vs-challenger comparison results, the final replacement decision, and benchmark configuration metadata. These artifacts were saved to:

```text
data/processed/modeling/model_benchmarking/